# 02_gold_personal - Spark SQL KPIs & Aggregations
Notebook này kiểm thử các câu truy vấn Spark SQL tính toán các chỉ số Top Tracks, Top Artists và Listening Schedules ở tầng Gold.

In [ ]:
# 1. Khởi tạo PySpark Session & Tạo Temp Views thử nghiệm
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Spotify_Gold_Local") \
    .getOrCreate()

# Đọc dữ liệu Silver local vừa làm sạch ở notebook trước
local_bronze = "data/bronze_spotify_raw/personal_*.json"
df_raw = spark.read.option("multiline", "true").json(local_bronze)
df_exploded = df_raw.select(F.explode("items").alias("item"), F.col("ingestion_metadata.ingestion_time"))
df_flat = df_exploded.select(
    F.to_timestamp(F.col("item.played_at")).alias("played_at"),
    F.col("item.track.id").alias("track_id"),
    F.col("item.track.name").alias("track_name"),
    F.col("item.track.duration_ms").cast("integer").alias("duration_ms"),
    F.col("item.track.album.id").alias("album_id"),
    F.explode("item.track.artists").alias("artist")
).select(
    "played_at", "track_id", "track_name", "duration_ms", "album_id",
    F.col("artist.id").alias("artist_id"), F.col("artist.name").alias("artist_name")
)

# Đăng ký Temp Views giả lập tầng Silver cho Spark SQL
df_flat.select("track_id", "track_name", "duration_ms", "album_id").dropDuplicates(["track_id"]).createOrReplaceTempView("dim_tracks")
df_flat.select("artist_id", "artist_name").dropDuplicates(["artist_id"]).createOrReplaceTempView("dim_artists")
df_flat.select("played_at", "track_id", "artist_id", "album_id").createOrReplaceTempView("fact_streams")

print("✅ Đã tạo các Temp Views giả lập tầng Silver thành công!")

In [ ]:
# 2. Kiểm thử SQL 1: Top Bài hát nghe nhiều nhất & Tổng phút nghe
gold_top_tracks = spark.sql("""
SELECT 
  t.track_id, 
  t.track_name, 
  a.artist_name, 
  COUNT(*) AS total_streams,
  ROUND(SUM(t.duration_ms) / 60000.0, 2) AS total_minutes_listened,
  DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS rank_by_streams
FROM fact_streams f
JOIN dim_tracks t ON f.track_id = t.track_id
JOIN dim_artists a ON f.artist_id = a.artist_id
GROUP BY 
  t.track_id, 
  t.track_name, 
  a.artist_name
ORDER BY 
  total_streams DESC
LIMIT 50
""")

print("🏆 Bảng Gold Top Tracks:")
gold_top_tracks.show(10, truncate=False)

In [ ]:
# 3. Kiểm thử SQL 2: Top Nghệ sĩ nghe nhiều nhất
gold_top_artists = spark.sql("""
SELECT 
  a.artist_id, 
  a.artist_name,
  COUNT(*) AS total_streams,
  ROUND(SUM(t.duration_ms) / 60000.0, 2) AS total_minutes_listened,
  DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS rank_by_streams
FROM fact_streams f
JOIN dim_artists a ON f.artist_id = a.artist_id
JOIN dim_tracks t ON f.track_id = t.track_id
GROUP BY a.artist_id, a.artist_name
ORDER BY total_streams DESC
LIMIT 50
""")

print("🎤 Bảng Gold Top Artists:")
gold_top_artists.show(10, truncate=False)

In [ ]:
# 4. Kiểm thử SQL 3: Phân bố khung giờ nghe nhạc
gold_schedule = spark.sql("""
SELECT 
  HOUR(f.played_at) AS hour_of_day,
  DAYOFWEEK(f.played_at) AS day_of_week_num,
  DATE_FORMAT(f.played_at, 'EEEE') AS day_of_week_name,
  CASE 
    WHEN HOUR(f.played_at) BETWEEN 5 AND 11 THEN 'Sáng (05h-12h)'
    WHEN HOUR(f.played_at) BETWEEN 12 AND 17 THEN 'Chiều (12h-18h)'
    WHEN HOUR(f.played_at) BETWEEN 18 AND 22 THEN 'Tối (18h-23h)'
    ELSE 'Đêm (23h-05h)'
  END AS time_slot,
  COUNT(*) AS stream_count
FROM fact_streams f
GROUP BY 
  HOUR(f.played_at),
  DAYOFWEEK(f.played_at),
  DATE_FORMAT(f.played_at, 'EEEE'),
  CASE 
    WHEN HOUR(f.played_at) BETWEEN 5 AND 11 THEN 'Sáng (05h-12h)'
    WHEN HOUR(f.played_at) BETWEEN 12 AND 17 THEN 'Chiều (12h-18h)'
    WHEN HOUR(f.played_at) BETWEEN 18 AND 22 THEN 'Tối (18h-23h)'
    ELSE 'Đêm (23h-05h)'
  END
""")

print("⏰ Bảng Gold Listening Schedule:")
gold_schedule.show(10, truncate=False)